# Hyperparameter tuning

In [1]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.entities import AmlCompute

from dotenv import load_dotenv
import os

load_dotenv("../.env", override=True)

subscription_id = os.environ["ARM_SUBSCRIPTION_ID"]
resource_group = os.environ["ARM_RESOURCE_GROUP"]
workspace = os.environ["AZ_ML_NAME"]
mlflow_tracking_uri = os.environ["MLFLOW_TRACKING_URI"]

ml_client = MLClient(
    DefaultAzureCredential(), subscription_id, resource_group, workspace
)

### Set up cluster

In [9]:
compute_name = "cpu-training-cluster"

try:
    cluster = AmlCompute(
        name=compute_name,
        size="STANDARD_DS3_v2",
        min_instances=0,
        max_instances=1,
        idle_time_before_scale_down=120,   # seconds
        tier="Dedicated",                  # or "LowPriority"
    )

    ml_client.compute.begin_create_or_update(cluster)

except Exception as e:
    raise e


### Train with a normal job

In [10]:
from azure.ai.ml import command, Input
from azure.ai.ml.constants import AssetTypes

# configure job
job = command(
    code="../mlops/hyperparameter_tuning/code/",
    inputs={
        "input_data": Input(type=AssetTypes.URI_FILE, path="azureml:diabetes-csv:1"),
        "regularization": 1,
        "scaler": "standard",
        "valid_size": 0.3
    },
    command="""python train.py \
                    --training_data ${{inputs.input_data}} \
                    --regularization ${{inputs.regularization}} \
                    --scaler ${{inputs.scaler}} \
                    --valid_size ${{inputs.valid_size}}
    """,
    environment="AzureML-sklearn-0.24-ubuntu18.04-py37-cpu@latest",
    compute=compute_name,
    display_name="train-model",
    experiment_name="train-classification-model"
    )


In [11]:
returned_job = ml_client.create_or_update(job)

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


### Sweep job for hyperparameter tuning

In [12]:
from azure.ai.ml.sweep import Choice


command_job_for_sweep = job(
    regularization=Choice(values=[0.01, 0.1, 1, 10]),
)

sweep_job = command_job_for_sweep.sweep(
    compute=compute_name,
    sampling_algorithm="grid",
    primary_metric="Accuracy",
    goal="Maximize",   
)

# Set the name 
sweep_job.experiment_name="sweep-job-hyperparameter-tuning"

# Set the limits
sweep_job.set_limits(
    max_total_trials=4,
    max_concurrent_trials=2,
    timeout=7200
    )

# Submit a sweep job
returned_sweep_job = ml_client.create_or_update(sweep_job)